In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Fonction Circuit IBM

> **Note:** * Les Qiskit Functions sont une fonctionnalité expérimentale disponible uniquement pour les utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles d'évoluer.

## Vue d'ensemble
La fonction Circuit IBM&reg; prend des [PUBs abstraits](./primitive-input-output) en entrée et renvoie des valeurs d'espérance atténuées en sortie. Cette fonction Circuit inclut un pipeline automatisé et personnalisé pour permettre aux chercheurs de se concentrer sur la découverte d'algorithmes et d'applications.

## Description
Après avoir soumis ton PUB, tes circuits abstraits et tes observables sont automatiquement transpilés, exécutés sur le matériel, et post-traités pour renvoyer des valeurs d'espérance atténuées. Pour ce faire, cette fonction combine les outils suivants :

- [Qiskit Transpiler Service](./qiskit-transpiler-service), incluant la sélection automatique de passes de transpilation basées sur l'IA et sur des heuristiques pour traduire tes circuits abstraits en circuits ISA optimisés pour le matériel
- [Suppression et atténuation des erreurs requises pour le calcul à l'échelle utilitaire](./error-mitigation-and-suppression-techniques), incluant le twirling de mesures et de portes, le découplage dynamique, la Twirled Readout Error eXtinction (TREX), la Zero-Noise Extrapolation (ZNE) et l'Amplification Probabiliste des Erreurs (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), pour exécuter les PUBs ISA sur le matériel et retourner des valeurs d'espérance atténuées

![Fonction Circuit IBM](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Démarrage rapide
Authentifie-toi avec ta [clé API](http://quantum.cloud.ibm.com/) et sélectionne la Qiskit Function comme suit. (Cet extrait suppose que tu as déjà [sauvegardé ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Exemple
Pour commencer, essaie cet exemple de base :

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Vérifie le [statut](/guides/functions#check-job-status) de ta charge de travail Qiskit Function ou récupère les [résultats](/guides/functions#retrieve-results) comme suit :

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Les résultats ont le même format qu'un [résultat Estimator](/guides/primitive-input-output#estimator-output) :

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Entrées
Le tableau suivant répertorie tous les paramètres d'entrée acceptés par la fonction Circuit IBM. La section [Options](#options) ci-dessous détaille les `options` disponibles.

| Nom      | Type                       | Description                                                                                                                                                                                                                         | Requis | Défaut                                                                  | Exemple                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Nom du backend sur lequel effectuer la requête.                                                                                                                                                                                              | Oui      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Un itérable d'objets de type PUB abstrait (primitive unified bloc), tels que des tuples `(circuit, observables)` ou `(circuit, observables, parameter_values)`. Voir [Vue d'ensemble des PUBs](/guides/primitive-input-output#overview-of-pubs) pour plus d'informations. Les circuits peuvent être abstraits (non-ISA). | Oui      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Options d'entrée. Voir la section **Options** pour plus de détails.                                                                                                                                                                                | Non       | Voir la section **Options** pour les détails.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Le nom de ressource cloud de l'instance à utiliser dans ce format.                                                                                                                                                                                        | Non       | Une instance est choisie aléatoirement si ton compte a accès à plusieurs instances. | `CRN`                   |

### Options
#### Structure
Comme pour les primitives Qiskit Runtime, les options de la fonction Circuit IBM peuvent être spécifiées sous forme de dictionnaire imbriqué. Les options couramment utilisées, telles que ``optimization_level`` et ``mitigation_level``, se trouvent au premier niveau. D'autres options plus avancées sont regroupées en différentes catégories, comme ``resilience``.

#### Valeurs par défaut
Si tu ne spécifies pas de valeur pour une option, la valeur par défaut définie par le service est utilisée.

#### Niveau d'atténuation
La fonction Circuit IBM prend également en charge `mitigation_level`. Le niveau d'atténuation spécifie la quantité de suppression et d'atténuation des erreurs à appliquer au job. Des niveaux plus élevés produisent des résultats plus précis, au prix de temps de traitement plus longs. Le degré de réduction des erreurs dépend de la méthode appliquée. Le niveau d'atténuation abstrait le choix détaillé des méthodes d'atténuation et de suppression des erreurs afin de permettre aux utilisateurs de raisonner sur le compromis coût/précision approprié à leur application. Le tableau suivant présente les méthodes correspondant à chaque niveau.

> **Note:** Bien que leurs noms soient similaires, `mitigation_level` applique des techniques différentes de celles utilisées par le `resilience_level` de l'Estimator.

Comme pour ``resilience_level`` dans Qiskit Runtime Estimator, ``mitigation_level`` spécifie un ensemble de base d'options soigneusement sélectionnées. Toutes les options que tu spécifies manuellement en complément du niveau d'atténuation s'appliquent en plus de cet ensemble de base. Par conséquent, tu pourrais en principe définir le niveau d'atténuation à 1 tout en désactivant l'atténuation des mesures, bien que cela ne soit pas conseillé.

| **Niveau d'atténuation** | **Technique** |
|:-:|:-:|
| 1 [Par défaut] | Découplage dynamique + twirling de mesures + TREX  |
| 2 | Niveau 1 + twirling de portes + ZNE via repliement de portes |
| 3 | Niveau 1 + twirling de portes + ZNE via PEA |

L'exemple suivant illustre la définition du niveau d'atténuation :

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Toutes les options disponibles
En plus de `mitigation_level`, la fonction Circuit IBM offre également un nombre limité d'options avancées permettant d'affiner le compromis coût/précision. Le tableau suivant présente toutes les options disponibles :

| Option | Sous-option | Sous-sous-option | Description | Choix | Défaut |
|-|-|-|-|-|-|
| default_precision |  |  | La précision par défaut à utiliser pour tout PUB ou appel `run()`<br/>qui n'en spécifie pas.<br/>Chaque PUB en entrée peut spécifier sa propre précision. Si la méthode `run()` reçoit une précision, cette valeur est utilisée pour tous les PUBs de l'appel `run()` qui ne spécifient pas la leur.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Temps d'exécution maximal en secondes, basé sur<br/>l'utilisation du QPU (et non sur le temps d'horloge). L'utilisation du QPU correspond au temps pendant lequel le QPU est dédié au traitement de ton job. Si un job dépasse cette limite de temps, il est annulé de force. | Nombre entier de secondes dans la plage [1, 10800] |  |
| mitigation_level |  |  | La quantité de suppression et d'atténuation des erreurs à appliquer. Consulte la section [Niveau d'atténuation](#mitigation-level) pour plus d'informations sur les méthodes utilisées à chaque niveau. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | La quantité d'optimisation à effectuer sur les circuits. [Des niveaux plus élevés](/guides/set-optimization) génèrent des circuits plus optimisés, au prix d'un temps de transpilation plus long. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Indique si le découplage dynamique doit être activé. Consulte [Techniques de suppression et d'atténuation des erreurs](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) pour l'explication de la méthode.  | True/False | True |
|  | sequence_type |  | La séquence de découplage dynamique à utiliser.<br/>* `XX` : utilise la séquence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm` : utilise la séquence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4` : utilise la séquence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Indique si le twirling de portes Clifford à 2 qubits doit être appliqué. | True/False | False |
|  | enable_measure |  | Indique si le twirling des mesures doit être activé. | True/False | True |
| resilience | measure_mitigation |  | Indique si la méthode d'atténuation des erreurs de mesure TREX doit être activée. Consulte [Techniques de suppression et d'atténuation des erreurs](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) pour l'explication de la méthode.  | True/False | True |
|  | zne_mitigation |  | Indique si la méthode d'atténuation des erreurs par extrapolation à zéro bruit (Zero Noise Extrapolation) doit être activée. Consulte [Techniques de suppression et d'atténuation des erreurs](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) pour l'explication de la méthode.  | True/False | False |
|  | zne | amplifier | La technique à utiliser pour amplifier le bruit. L'une des options suivantes : <br/> - `gate_folding` (par défaut) utilise le repliement de portes à 2 qubits pour amplifier le bruit. Si le facteur de bruit nécessite d'amplifier seulement un sous-ensemble de portes, celles-ci sont choisies aléatoirement.<br/><br/> - `gate_folding_front` utilise le repliement de portes à 2 qubits pour amplifier le bruit. Si le facteur de bruit nécessite d'amplifier seulement un sous-ensemble de portes, celles-ci sont sélectionnées depuis le début du circuit DAG trié topologiquement.<br/><br/> - `gate_folding_back` utilise le repliement de portes à 2 qubits pour amplifier le bruit. Si le facteur de bruit nécessite d'amplifier seulement un sous-ensemble de portes, celles-ci sont sélectionnées depuis la fin du circuit DAG trié topologiquement.<br/><br/> - `pea` utilise une technique appelée Amplification Probabiliste des Erreurs (PEA) pour amplifier le bruit. Consulte [Techniques de suppression et d'atténuation des erreurs](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) pour l'explication de la méthode.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Facteurs de bruit à utiliser pour l'amplification du bruit. | liste de flottants ; chaque flottant >= 1 | (1, 1.5, 2) pour PEA, et (1, 3, 5) sinon. |
|  |  | extrapolator | Facteurs de bruit auxquels évaluer les modèles d'extrapolation ajustés. Cette option n'affecte ni l'exécution ni l'ajustement du modèle ; elle détermine uniquement les points auxquels les objets `extrapolator` sont évalués pour être renvoyés dans les champs de données `evs_extrapolated` et `stds_extrapolated`. | un ou plusieurs parmi `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Indique si la méthode d'atténuation des erreurs par Annulation Probabiliste des Erreurs (Probabilistic Error Cancellation) doit être activée. Consulte [Techniques de suppression et d'atténuation des erreurs](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) pour l'explication de la méthode.  | True/False | False |
|  | pec | max_overhead | Le surcoût maximal d'échantillonnage de circuits autorisé, ou `None` pour aucune limite. | None / entier > 1 | 100 |

Dans l'exemple suivant, définir le niveau d'atténuation à 1 désactive initialement l'atténuation ZNE, mais définir `zne_mitigation` à `True` remplace la configuration correspondante issue de `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Sorties
La sortie d'une fonction Circuit est un [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), qui contient deux champs :

- Un ou plusieurs objets [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Ceux-ci peuvent être indexés directement depuis le `PrimitiveResult`.

- Des métadonnées au niveau du job.

Chaque `PubResult` contient un champ `data` et un champ `metadata`.

- Le champ `data` contient au minimum un tableau de valeurs d'espérance (`PubResult.data.evs`) et un tableau d'erreurs standard (`PubResult.data.stds`). Il peut également contenir d'autres données selon les options utilisées.

- Le champ `metadata` contient les métadonnées au niveau du PUB (`PubResult.metadata`).

L'extrait de code suivant décrit le format du `PrimitiveResult` (et du `PubResult` associé).